In [ ]:
# ======================================================================
# CLEAN RENT DATA: RTB/ESRI rent index, ready for the temporal analysis
# ======================================================================
# Input:  processed_dataset\rent_index_dataset\rent_2021_onwards.csv
# Output: clean_dataset\rent\rent_clean.csv          (all rows)
#         clean_dataset\rent\rent_county_headline.csv (26 counties x 20 quarters)
#
# What this is for (from the proposal, Student 3):
#   - temporal pattern analysis of county prices and rents
#   - context for the house price model (rent per county per quarter)
#
# These are published averages, not individual tenancies, so there is very
# little to clean. Mainly: matching the county names to the shared list and
# making the small headline table.
#
# Needs 00_county_lookup.ipynb to be run first (uses county_lookup.csv)
# ======================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

pd.set_option('display.max_columns', None)

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project\dataSet'

rent_file = base + r'\processed_dataset\rent_index_dataset\rent_2021_onwards.csv'
clean     = base + r'\clean_dataset'

In [ ]:
rent = pd.read_csv(rent_file)

print(rent.shape)
rent.head()

In [ ]:
# Quick checks: spaces in text, missing values, duplicates

for col in ['quarter', 'bedrooms', 'property_type', 'location', 'county', 'location_level']:
    rent[col] = rent[col].str.strip()

print('Missing values:')
print(rent.isnull().sum())
print()
key = ['quarter', 'location', 'bedrooms', 'property_type']
print('Duplicate rows:', rent.duplicated(subset=key).sum())

In [ ]:
# County: the processed notebook already made a county column, but here the
# shared county_lookup.csv is used so the county_id matches BER, planning and
# PPR. Checking both versions agree.

lookup = pd.read_csv(clean + r'\county_lookup.csv')
lookup = lookup[lookup['source'] == 'rent'][['raw_name', 'county', 'county_id']]
lookup = lookup.rename(columns={'county': 'county_lookup'})

rent = rent.merge(lookup, left_on='location', right_on='raw_name', how='left')
rent = rent.drop(columns=['raw_name'])

print('Rows without a county:', rent['county_lookup'].isnull().sum())          # should be 0
print('Rows where the two versions differ:', (rent['county'] != rent['county_lookup']).sum())   # should be 0

rent['county'] = rent['county_lookup']
rent = rent.drop(columns=['county_lookup'])

In [ ]:
# Checking the rent values look sensible (monthly rent in euro)

print(rent['avg_rent'].describe().round(0))
print('Rents below 100 or above 10,000:', ((rent['avg_rent'] < 100) | (rent['avg_rent'] > 10000)).sum())

In [ ]:
# Headline flag: the row for the whole county, all bedrooms, all property
# types. This is the single 'average rent' number per county per quarter
# used for the temporal analysis and to join with house prices.

rent['is_county_headline'] = 0
rent.loc[(rent['location_level'] == 'county') &
         (rent['bedrooms'] == 'All bedrooms') &
         (rent['property_type'] == 'All property types'), 'is_county_headline'] = 1

print('Headline rows:', rent['is_county_headline'].sum())   # 26 counties x 20 quarters = 520

In [ ]:
# Small headline table: one row per county per quarter

headline = rent[rent['is_county_headline'] == 1]
headline = headline[['county', 'county_id', 'quarter', 'year', 'quarter_no', 'avg_rent']]
headline = headline.sort_values(['county', 'quarter']).reset_index(drop=True)

# Every county should have all 20 quarters (2021Q1 to 2025Q4)
quarters_per_county = headline.groupby('county')['quarter'].count()
print('Counties:', headline['county'].nunique())
print('Quarters per county (min / max):', quarters_per_county.min(), '/', quarters_per_county.max())

headline.head()

In [ ]:
# Rent change per county from 2021Q1 to 2025Q4, as a quick sense check

first = headline[headline['quarter'] == '2021Q1'].set_index('county')['avg_rent']
last = headline[headline['quarter'] == '2025Q4'].set_index('county')['avg_rent']

change = ((last - first) / first * 100).round(1).sort_values()

plt.figure(figsize=(10, 7))
change.plot(kind='barh')
plt.xlabel('% change in average rent, 2021Q1 to 2025Q4')
plt.title('Rent growth by county')
plt.show()

In [ ]:
# Column order for the full clean table: keys first, then details, then rent

rent_clean = rent[['county', 'county_id', 'location', 'location_level', 'quarter', 'year',
                   'quarter_no', 'bedrooms', 'property_type', 'is_county_headline', 'avg_rent']]

print(rent_clean.shape)
rent_clean.info()

In [ ]:
# Saving both tables.

save_folder = clean + r'\rent'
os.makedirs(save_folder, exist_ok=True)

rent_clean.to_csv(save_folder + r'\rent_clean.csv', index=False)
headline.to_csv(save_folder + r'\rent_county_headline.csv', index=False)

print('Saved', rent_clean.shape, 'to', save_folder + r'\rent_clean.csv')
print('Saved', headline.shape, 'to', save_folder + r'\rent_county_headline.csv')